In [1]:
import os
import faiss
import kagglehub

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

c:\Users\devas\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


data preparation

In [2]:
path = kagglehub.dataset_download("kshitizregmi/jobs-and-job-description")
data = pd.read_csv(os.path.join(path, 'job_title_des.csv'))

model training

In [3]:
open_from = './mern deployment/prediction'

model = SentenceTransformer(f"{open_from}/model.pkl")
embs = np.load(f"{open_from}/embs.npy")
data = pd.read_csv(f"{open_from}/data.csv")
indices = faiss.IndexFlatL2(embs.shape[1])
indices.add(embs)

FileNotFoundError: Path ./mern deployment/prediction/model.pkl not found

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embs = model.encode(data['Job Description'], show_progress_bar=True)

indices = faiss.IndexFlatL2(embs.shape[1])
indices.add(embs)

In [ ]:
def find_match(state, n=5):
    emb = model.encode([state])
    _, idx = indices.search(emb, 5)
    res = []
    for i in idx[0]:
        r = data.iloc[i]
        res.append({
            'Job Title': r['Job Title'],
            'Job Description': r['Job Description'],
        })
    return res

In [ ]:
match = find_match('part time')
print(match)

model evaluation

model deployment

In [ ]:
save_to = 'mern deployment/prediction'

model.save(f"{save_to}/model.pkl")
np.save(f"{save_to}/embs.npy", embs)
data.to_csv(f"{save_to}/data.csv", index=False)